In [11]:
import os
import subprocess
from pathlib import Path
import nbformat
from nbconvert import PythonExporter

# Define paths
base_dir = Path.home() / "drg-pipeline" / "data-cleaning"
debug_dir = base_dir / "debug" / "codebase"
temp_dir = debug_dir / "temp"
r_scripts_dir = base_dir / "r_scripts_v2"
instructions_path = debug_dir / "instructions.md"

# Ensure temp directory exists
temp_dir.mkdir(parents=True, exist_ok=True)

# List of Jupyter notebooks to convert
notebooks = [
    "00b-drg-partial.ipynb",
    "02b-drg-grouping-py-v2.ipynb",
    "01-drg-cleaning-v2.ipynb",
    "02-drg-grouping-v2.ipynb"
]

# Read & clean R scripts
def clean_lines(lines):
    return [line for line in lines if line.strip() and not line.strip().startswith("#")]

# Start writing to instructions.md
with open(instructions_path, "w", encoding="utf-8") as f:
    f.write("# Codebase Context\n\n")

    # Process helper R scripts
    f.write("## Helper Scripts\n")
    for r_file in sorted(r_scripts_dir.glob("*.R")):
        f.write(f"\n### {r_file.name}\n```r\n")
        f.write("\n".join(clean_lines(r_file.read_text().splitlines())) + "\n```\n")

    # Process main R scripts (from Jupyter notebooks)
    f.write("\n## R Notebooks\n")
    for nb in notebooks:
        nb_path = base_dir / nb
        if nb_path.exists():
            with open(nb_path, "r", encoding="utf-8") as nb_file:
                nb_content = nbformat.read(nb_file, as_version=4)

            # Detect if it's an R notebook
            if nb_content.get("metadata", {}).get("kernelspec", {}).get("language", "") == "R":
                f.write(f"\n### {nb_path.name}\n```r\n")
                r_code = "\n\n".join(
                    cell["source"].strip() for cell in nb_content["cells"] if cell["cell_type"] == "code"
                )
                f.write("\n".join(clean_lines(r_code.splitlines())) + "\n```\n")

    # Process main Python script (from Jupyter notebook)
    f.write("\n## Python Notebook\n")
    for nb in notebooks:
        nb_path = base_dir / nb
        if nb_path.exists():
            with open(nb_path, "r", encoding="utf-8") as nb_file:
                nb_content = nbformat.read(nb_file, as_version=4)

            # Detect if it's a Python notebook
            if nb_content.get("metadata", {}).get("kernelspec", {}).get("language", "") == "python":
                f.write(f"\n### {nb_path.name}\n```python\n")
                script_content, _ = PythonExporter().from_notebook_node(nb_content)
                f.write("\n".join(clean_lines(script_content.splitlines())) + "\n```\n")

# Cleanup temp directory
subprocess.run(["rm", "-rf", str(temp_dir)])

print(f"Instructions saved to: {instructions_path}")


Instructions saved to: /home/resurreccion_cmc/drg-pipeline/data-cleaning/debug/codebase/instructions.md
